In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


In [2]:
PROCESSED_DIR = Path("../data/processed")
SELECTED_PM25 = PROCESSED_DIR / "pm25_selected_station_days.parquet"
SELECTED_PM10 = PROCESSED_DIR / "pm10_selected_station_days.parquet"

TARGET_COLUMNS = {"pm25": "pm25", "pm10": "pm10"}
LAGGED_DAYS = (1, 2, 3)
ROLLING_WINDOWS = (3, 7)
HIGH_LIMITS = {"pm25": 25, "pm10": 50}

pm25_df = pd.read_parquet(SELECTED_PM25)
pm10_df = pd.read_parquet(SELECTED_PM10)
print(f"PM2.5 selected rows: {len(pm25_df):,}")
print(f"PM10 selected rows: {len(pm10_df):,}")


PM2.5 selected rows: 3,995
PM10 selected rows: 8,798


In [3]:
def build_daily_features(df, pollutant):
    df = df.sort_values(["station_id", "date"]).copy()
    df["date"] = pd.to_datetime(df["date"])

    parts = []
    for station_id, group in df.groupby("station_id", sort=False):
        station_name = group["station_name"].iloc[0]
        group = group.sort_values("date").set_index("date")
        full_index = pd.date_range(group.index.min(), group.index.max(), freq="D")
        group = group.reindex(full_index)
        group["station_id"] = station_id
        group["station_name"] = station_name

        target = group[pollutant]

        for lag in LAGGED_DAYS:
            group[f"{pollutant}_lag_{lag}"] = target.shift(lag)

        for window in ROLLING_WINDOWS:
            group[f"{pollutant}_rolling_{window}"] = target.shift(1).rolling(window).mean()
            group[f"{pollutant}_rolling_std_{window}"] = target.shift(1).rolling(window).std()

        group["date"] = group.index
        parts.append(group)

    result = pd.concat(parts, ignore_index=True).reset_index(drop=True)
    result = result[result[pollutant].notna()].copy()
    result = result.sort_values(["station_id", "date"]).reset_index(drop=True)

    result["month"] = result["date"].dt.month
    result["day_of_week"] = result["date"].dt.dayofweek
    result["year"] = result["date"].dt.year
    result["month_sin"] = np.sin(2 * np.pi * result["month"] / 12)
    result["month_cos"] = np.cos(2 * np.pi * result["month"] / 12)
    result["dow_sin"] = np.sin(2 * np.pi * result["day_of_week"] / 7)
    result["dow_cos"] = np.cos(2 * np.pi * result["day_of_week"] / 7)

    result[f"{pollutant}_high"] = (result[pollutant] > HIGH_LIMITS[pollutant]).astype(int)

    return result


features_pm25 = build_daily_features(pm25_df, "pm25")
features_pm10 = build_daily_features(pm10_df, "pm10")
print(f"Features PM2.5 rows: {len(features_pm25):,}")
print(f"Features PM10 rows: {len(features_pm10):,}")


Features PM2.5 rows: 3,995
Features PM10 rows: 8,798


In [4]:
core_feature_cols = [
    "pm25_lag_1",
    "pm25_lag_2",
    "pm25_lag_3",
    "pm25_rolling_3",
    "pm25_rolling_7",
]


def usable_rows(features, feature_cols):
    return len(features.dropna(subset=feature_cols))


print("PM2.5")
print("  total rows:", len(features_pm25))
print("  rows with all lag/rolling features (usable):", usable_rows(features_pm25, core_feature_cols))
print("  high-pollution rows:", int(features_pm25["pm25_high"].sum()))

display(features_pm25[["station_name", "date", "pm25", "pm25_lag_1", "pm25_rolling_3", "pm25_rolling_7", "pm25_high"]].head(15))


PM2.5
  total rows: 3995
  rows with all lag/rolling features (usable): 1912
  high-pollution rows: 1311


,station_name,date,pm25,pm25_lag_1,pm25_rolling_3,pm25_rolling_7,pm25_high
0,Gazi Baba,2023-06-27,9.475000,NaN,NaN,NaN,0
1,Gazi Baba,2023-06-28,11.458333,9.475000,NaN,NaN,0
2,Gazi Baba,2023-06-29,6.600000,11.458333,NaN,NaN,0
3,Gazi Baba,2023-06-30,10.837500,6.600000,9.177778,NaN,0
4,Gazi Baba,2023-07-01,10.291667,10.837500,9.631944,NaN,0
5,Gazi Baba,2023-07-02,7.208333,10.291667,9.243056,NaN,0
6,Gazi Baba,2023-07-03,9.783333,7.208333,9.445833,NaN,0
7,Gazi Baba,2023-07-04,7.033333,9.783333,9.094444,9.379167,0
8,Gazi Baba,2023-07-05,7.812500,7.033333,8.008333,9.030357,0
9,Gazi Baba,2023-07-06,5.562500,7.812500,8.209722,8.509524,0


In [5]:
core_feature_cols_10 = [
    "pm10_lag_1",
    "pm10_lag_2",
    "pm10_lag_3",
    "pm10_rolling_3",
    "pm10_rolling_7",
]

print("PM10")
print("  total rows:", len(features_pm10))
print("  rows with all lag/rolling features (usable):", usable_rows(features_pm10, core_feature_cols_10))
print("  high-pollution rows:", int(features_pm10["pm10_high"].sum()))

display(features_pm10[["station_name", "date", "pm10", "pm10_lag_1", "pm10_rolling_3", "pm10_rolling_7", "pm10_high"]].head(15))


PM10
  total rows: 8798
  rows with all lag/rolling features (usable): 3575
  high-pollution rows: 2694


,station_name,date,pm10,pm10_lag_1,pm10_rolling_3,pm10_rolling_7,pm10_high
0,Gazi Baba,2016-12-11,334.590909,NaN,NaN,NaN,1
1,Gazi Baba,2016-12-12,187.505263,334.590909,NaN,NaN,1
2,Gazi Baba,2016-12-13,16.307778,187.505263,NaN,NaN,0
3,Gazi Baba,2016-12-14,115.007273,16.307778,179.467983,NaN,1
4,Gazi Baba,2016-12-15,146.438889,115.007273,106.273438,NaN,1
5,Gazi Baba,2016-12-16,69.547619,146.438889,92.584646,NaN,1
6,Gazi Baba,2016-12-17,151.984211,69.547619,110.331260,NaN,1
7,Gazi Baba,2016-12-18,248.954545,151.984211,122.656906,145.911706,1
8,Gazi Baba,2017-02-19,63.933333,NaN,NaN,NaN,1
9,Gazi Baba,2017-02-20,81.367083,63.933333,NaN,NaN,1


In [6]:
FEATURE_OUT_PM25 = PROCESSED_DIR / "features_pm25.parquet"
FEATURE_OUT_PM10 = PROCESSED_DIR / "features_pm10.parquet"

features_pm25.to_parquet(FEATURE_OUT_PM25, index=False)
features_pm10.to_parquet(FEATURE_OUT_PM10, index=False)
print(f"Saved: {FEATURE_OUT_PM25.name} ({len(features_pm25):,} rows, {features_pm25.shape[1]} cols)")
print(f"Saved: {FEATURE_OUT_PM10.name} ({len(features_pm10):,} rows, {features_pm10.shape[1]} cols)")


Saved: features_pm25.parquet (3,995 rows, 88 cols)
Saved: features_pm10.parquet (8,798 rows, 88 cols)


In [7]:
print("New feature columns added for PM2.5:")
print([c for c in features_pm25.columns if c.startswith("pm25_lag") or c.startswith("pm25_rolling") or c in ("month_sin", "month_cos", "dow_sin", "dow_cos", "pm25_high")])
print()
print("New feature columns added for PM10:")
print([c for c in features_pm10.columns if c.startswith("pm10_lag") or c.startswith("pm10_rolling") or c in ("month_sin", "month_cos", "dow_sin", "dow_cos", "pm10_high")])


New feature columns added for PM2.5:
['pm25_lag_1', 'pm25_lag_2', 'pm25_lag_3', 'pm25_rolling_3', 'pm25_rolling_std_3', 'pm25_rolling_7', 'pm25_rolling_std_7', 'month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'pm25_high']

New feature columns added for PM10:
['pm10_lag_1', 'pm10_lag_2', 'pm10_lag_3', 'pm10_rolling_3', 'pm10_rolling_std_3', 'pm10_rolling_7', 'pm10_rolling_std_7', 'month_sin', 'month_cos', 'dow_sin', 'dow_cos', 'pm10_high']
